# Глава 15. Фреймворки

Для начала рассмотрим инструменты для локального запуска моделей.

## Дескптопные решения

#### llama.cpp
[[llama.cpp]](https://github.com/ggml-org/llama.cpp) - появившийся в 2023 году легковесный фреймворк и утилита для инференса локальных языковых моделей на десктопном железе, альтернатива тяжелым нейросетевым SDK типа PyTorch. Детище болгарского разработчика Георгия Герганова, написанное на C++, которое получило признание в сообществе инженеров и для которого позже появились UI обертки, самые изветсные LM Studio и Ollama. На момент 2023 года, если разработчик хотел использовать возможности языковых моделей, выбор у него стоял между подключением платного API для облачных моделей типа GPT-4, и покупкой дорогой видеокарты на 24ГБ для скромных 7B мод. Целью Герганова был запуск только что вышедшей модели LLAMA-1 на своём макбуке. У него был написанный с нуля свой движок тензорных вычислений GGML, в котором он убрал все ненужные зависимости, оставив только мимнимальный набор команд для инференса.

Библиотека работает с жестко заданным списком доступных архитекутур (LLAMA, Qwen, Gemma и других), веса моделей должны быть скачены локально, в формате GGUF. В отличие от продакшен систем, ориентированных на параллельное вычисление CUDA ядрами, llama.cpp не требует загрузки целиком в память и, что главное, умеет работать с весами произвольной битности. Кроме того, это не серверное решение, оно не опитмизировано под поток запросов и не имеет хитрых механизмов батчинга, распредленного внимания и т.п., зато очень хорошо экономит память.

Можно использовать как утилиту, как API (есть команда `llamaserver`), как SDK: вызывать из кода на С++ или Python проектов. Современем добавили феукционал типа скачивания моделей из хаба HugginfaceF и сервинга HTTP зарпосов, но основая задачей остается инференс на потребительском железе. Из коробки бибилотека поддерживает спекулятивное декодирование, взяли три базовых режима: с внешней моделью, с внутренней моделью и с промптом. Из коробки RAG не поддерживается, но можно собрать - первый сервер запускается в режиме эмбедингов (`llamaserver --embeddings`), второй - модель для генерации ответа

#### GGML
Фундамент llama.cpp - это фреймворк тензорных вычислений GGML. Греганов написал его годом ранее в 2022. Написал его для модели Whisper - популярной модели рапознавания речи от OpenAI. вдозновившись минимализмом утилит, FFmpeg. Исходный код 1MB.

#### LM Studio
Спустя пару месяцев на основе библиотеки llama.cpp компания ElementLabs создала удобный пользовательский интерфейс, 

#### Ollama
В июле инженеры (ex-Docker) выпустили обёртку для llama.cpp, которая добавила удобное упарвление моделями в стиле Docker (), создали репозиторий моделей, и сделали более умный механизм использования памяти видеокарты.

#### AnythingLLM
Решение от Mintplex, вышедшее в 2024 году. Десктопное и серверное open-source приложение, ставшее стандартом для построения локальных RAG-систем (Retrieval-Augmented Generation) без написания кода. 
Если LM Studio и Ollama сфокусированы на самом чате, то AnythingLLM берет на себя всю сложную инфраструктуру работы со знаниями. 
Приложение «из коробки» содержит парсеры документов (PDF, Office, TXT), встроенную легковесную векторную базу данных (LanceDB) и алгоритмы поиска. Оно позволяет пользователю в пару кликов загрузить корпоративные файлы, векторизовать их и подключить к локальной языковой модели через Ollama или LM Studio. В результате получается полностью приватный умный ассистент, отвечающий строго по базе документов компании.

## Серверные решения

#### vLLM
Полноценный высокопроизводительный движок инференса промышленного уровня (production-ready), разработанный исследователями из Беркли. Если движки семейства llama.cpp создавались для экономии памяти на домашних ПК, то главная цель vLLM — максимизация пропускной способности (throughput) и скорости генерации на мощных серверных GPU (NVIDIA A100, H100, RTX 3090/4090). Фреймворк ориентирован на сценарии, когда к одной модели одновременно обращаются десятки, сотни или тысячи пользователей (веб-сервисы, корпоративные чат-боты, API-облака).

Главным технологическим прорывом vLLM стало внедрение алгоритма PagedAttention. 
В классических системах память под контекст диалога (KV-кэш) выделялась монолитным статическим блоком, что приводило к фрашгментации VRAM памяти на видеокарте.

PagedAttention управляет памятью динамически, разбивая кэш на небольшие «страницы», аналогично виртуальной памяти в операционных системах. 
Это позволило vLLM снизить фрагментацию памяти практически до нуля и обрабатывать входящие запросы гигантскими динамическими пакетами (Continuous Batching). 
Фреймворк жестко завязан на экосистему CUDA и Triton, требует полной загрузки модели в видеопамять и не оптимизирован под работу на обычных CPU, что делает его де-факто стандартом для серверного бэкенда, но избыточным решением для индивидуального домашнего использования.

## Бекенды

